In [1]:
import time
from pathlib import Path
from github_rag.ingestion.repository import GitHubRepositoryLoader
from github_rag.processing.processor import Processor
from github_rag.chunking.chunker import Chunker
from github_rag.embedding.embedding_model import create_embedding_model
from github_rag.vectordatabase.chroma_store import ChromaStore
from github_rag.retrieval.retriever import Retriever

c:\Users\david\anaconda3\envs\github-rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
REPOSITORY_URL = "https://github.com/davidfernxndez/spark-active-learning"

## 1. Ingestion -> processing -> chunking

This section apply the Ingestion, processing and chunking phase to get the prepared *LangChain Documents* for Embedding and VectorDB storing.

In [3]:
# Set the Loader Object with the URL
loader = GitHubRepositoryLoader(REPOSITORY_URL)

# Load the repository files as a list of LangChain Documents
documents = loader.load()

print(f"Number of documents ingested: {len(documents)}")

Number of documents ingested: 12


In [4]:
# Set the Processor Object
processor = Processor()

# Apply processing to the ingested documents
documents_processed = processor.process(documents)

print(f"Number of documents before processing: {len(documents_processed)}")

Number of documents before processing: 45


In [5]:
# Set the Chunker object that managed all the specific chunkers
chunker = Chunker()

# Apply chunking
documents_chunking = chunker.chunk(documents_processed)

print(f"Number of documents before chunking: {len(documents_chunking)}")

Number of documents before chunking: 183


## 2. Load the Embedding model

We first need to load the embedding model that will be passed to the ChromaDB instance. The same model used to generate the document embeddings must be used to embed user queries during the retrieval process.

In [6]:
# Set the HuggingFace embedding model name
embedding_model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

# Load the model
embedding_model = create_embedding_model(embedding_model_name)

## 3. Create Chroma Database

The `ChromaStore` class supports two different storage modes depending on the execution environment. For local development, it can create a **persistent Chroma database** stored on disk. The database is stored in the `/data` directory at the root of the repository, allowing it to persist between executions and making it convenient for development and experimentation.

It also supports creating an **in-memory Chroma database**, which is intended for the Streamlit deployment. In this mode, the vector database exists only during the current application session and does not require persistent storage on disk.

In [7]:
start_time = time.time()
# Create ChromaDB instance for local disk storing
vectorDB_local = ChromaStore(
    mode="local",
    collection_name="pruebas"
)

# Create the database with documents and the embedding model
vectorDB_local.create(
    documents=documents_chunking,
    embedding_model=embedding_model
)
print(f"Database generation time: {time.time() - start_time:.2f} seconds")

# Check the number of stored documents
stored_documents = vectorDB_local.count()

print(f"Documents provided: {len(documents_chunking)}")
print(f"Documents stored:   {stored_documents}")

Database generation time: 4.56 seconds
Documents provided: 183
Documents stored:   183


In [8]:
file_paths = vectorDB_local.get_unique_file_paths()

print(file_paths)

['README.md', 'data/data_info.md', 'enviroment.yml', 'main.ipynb', 'performance_experiment.ipynb', 'scalability_experiments.ipynb', 'scripts/AL_methods.py', 'scripts/AL_performance.py', 'scripts/AL_scalability.py', 'scripts/config.py', 'scripts/dowload_data.py', 'scripts/plot_results.py']


## 4. Retrieval

In [9]:
# Query
query = "Where is broadcast performed?"
print("=" * 100)
print("RETRIEVAL COMPARISON")
print("=" * 100)
print(f"\nQuery: {query}\n")


# ============================================================
# 1. Similarity Search
# ============================================================

results_similarity = vectorDB_local.similarity_search_with_score(
    query=query,
    k=5
)


print("=" * 100)
print("1. SIMILARITY SEARCH")
print("=" * 100)

for i, (document, distance) in enumerate(results_similarity, start=1):

    print(f"\n--- Result {i} ---")
    print(f"Distance : {distance:.4f}")
    print(f"Source   : {document.metadata.get('source')}")
    print(f"File     : {document.metadata.get('file_path')}")
    print(f"\n{document.page_content}")


# ============================================================
# 2. Maximal Marginal Relevance (MMR)
# ============================================================

results_mmr = vectorDB_local.max_marginal_relevance_search(
    query=query,
    k=5,
    fetch_k=20,
    lambda_mult=0.5
)


print("\n\n")
print("=" * 100)
print("2. MAXIMAL MARGINAL RELEVANCE (MMR)")
print("=" * 100)

print("\nParameters:")
print("  k           = 5")
print("  fetch_k     = 20")
print("  lambda_mult = 0.5")

for i, document in enumerate(results_mmr, start=1):

    print(f"\n--- Result {i} ---")
    print(f"Source : {document.metadata.get('source')}")
    print(f"File   : {document.metadata.get('file_path')}")
    print(f"\n{document.page_content}")


# ============================================================
# 3. Summary
# ============================================================

print("\n\n")
print("=" * 100)
print("SUMMARY")
print("=" * 100)

print("\nSimilarity Search:")
for i, (document, distance) in enumerate(results_similarity, start=1):
    print(
        f"{i}. "
        f"{document.metadata.get('file_path')} "
        f"(distance={distance:.4f})"
    )

print("\nMMR:")
for i, document in enumerate(results_mmr, start=1):
    print(
        f"{i}. "
        f"{document.metadata.get('file_path')}"
    )

RETRIEVAL COMPARISON

Query: Where is broadcast performed?

1. SIMILARITY SEARCH

--- Result 1 ---
Distance : 32.3970
Source   : https://github.com/davidfernxndez/spark-active-learning/blob/main/scripts/AL_methods.py
File     : scripts/AL_methods.py

Finally, the IDs of the B selected instances are broadcast to all worker 
    nodes, while the full train_df dataset remains in-place without being 
    shuffled across the network. 

    A Broadcast Hash Join is performed to identify the target instances, 
    simulating the active learning oracle labeling step by updating their 
    state from ``U`` (unlabeled) to ``L`` (labeled). Broadcasting the small 
    set of selected IDs  instead of performing a full Shuffle Hash 
    or Sort-Merge Join reduces network I/O  and execution latency.

    Parameters
    ----------
    candidates_df : pyspark.sql.DataFrame
        DataFrame containing the unlabeled candidates instances with the columns
        ``id_sample``, ``features``, and ``uncerta

In [10]:
retriever = Retriever(vectorDB_local)



In [100]:
query = "Which MLlib evaluator is used to define the metric in scripts/AL_methods.py?"
results = retriever.retrieve(query=query, available_files=file_paths)

In [101]:
results

[Document(id='f7d00113-3c6a-4caf-9f30-c4a972a1a410', metadata={'chunk_index': 9, 'file_type': '.py', 'branch': 'main', 'file_path': 'scripts/AL_methods.py', 'source': 'https://github.com/davidfernxndez/spark-active-learning/blob/main/scripts/AL_methods.py', 'repository': 'davidfernxndez/spark-active-learning', 'chunk_id': 'scripts/AL_methods.py::9'}, page_content='# Train the model using the labeled subset\n    model_fit = model.fit(labeled_train_df)\n\n    # Generate predictions for the test set.\n    test_pred_df = model_fit.transform(test_df)\n\n    # Evaluate the predictions using the specified classification metric.\n    evaluator = MulticlassClassificationEvaluator(\n        labelCol="label",\n        predictionCol="prediction",\n        metricName=metric_name\n    )\n    metric_value = evaluator.evaluate(test_pred_df)\n    \n    print(f"\\t- Model evaluation completed -> {metric_name} = {metric_value:.4f} ") \n\n    return model_fit, metric_value'),
 Document(id='65947807-6705-4

In [102]:
context = "\n\n".join(
    f"""File_name: {doc.metadata.get("file_path")}
Source: {doc.metadata.get("source")}

Content:
{doc.page_content}"""
    for doc in results
)

prompt = f"""
You are an assistant specialized in answering questions
about GitHub repositories.

Use the retrieved context to answer the user's question.

Context:
{context}

Question:
{query}

Instructions:
- Base your answer on the relevant retrieved context.
- If the context does not contain enough information, explicitly state that
  rather than guessing.
- Cite at the end the source file(s) used.
- Format each citation as a Markdown link using the file name as the link text:
  Source: [<File_name>](<Source>)
"""

print(prompt)


You are an assistant specialized in answering questions
about GitHub repositories.

Use the retrieved context to answer the user's question.

Context:
File_name: scripts/AL_methods.py
Source: https://github.com/davidfernxndez/spark-active-learning/blob/main/scripts/AL_methods.py

Content:
# Train the model using the labeled subset
    model_fit = model.fit(labeled_train_df)

    # Generate predictions for the test set.
    test_pred_df = model_fit.transform(test_df)

    # Evaluate the predictions using the specified classification metric.
    evaluator = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName=metric_name
    )
    metric_value = evaluator.evaluate(test_pred_df)
    
    print(f"\t- Model evaluation completed -> {metric_name} = {metric_value:.4f} ") 

    return model_fit, metric_value

File_name: scripts/AL_methods.py
Source: https://github.com/davidfernxndez/spark-active-learning/blob/main/scripts/AL_method

In [30]:
from langchain_ollama import ChatOllama

print("langchain-ollama importado correctamente")

langchain-ollama importado correctamente


In [31]:
llm = ChatOllama(
    model="qwen3:1.7b",
    temperature=0,
)

In [86]:
import time

start = time.time()

response = llm.invoke(prompt)

print(f"Time: {time.time() - start:.2f} seconds")
print(response.content)

Time: 13.39 seconds
Data leakage prevention during feature standardization is achieved by fitting the `StandardScaler` exclusively on the training data, as explicitly stated in the context. This ensures that the scaling parameters (mean and standard deviation) are computed from the training set, preventing the test set from being inadvertently scaled using training data, which could introduce bias. 

Source:  
Source: [scripts/AL_methods.py](https://github.com/davidfernxndez/spark-active-learning/blob/main/scripts/AL_methods.py)


In [ ]:
import time

# Insert API KEY from .env

from langchain_groq import ChatGroq
"""
llm_groq = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=GROQ_API_KEY,
    temperature=0,
)
"""

In [103]:
response = llm_groq.invoke(prompt)

print(response.content)

The metric is defined using **`MulticlassClassificationEvaluator`** from Spark MLlib.  

Source: [scripts/AL_methods.py](https://github.com/davidfernxndez/spark-active-learning/blob/main/scripts/AL_methods.py)
